# **Lakehouse with MobilityDB as engine** (open Parquet, *no* index)

MobilityDB used purely as an *engine over the open lakehouse files*: the L0 Parquet is loaded into a
plain heap (trajectory kept as EWKB `bytea`), and each query decodes it on the fly with a manual
bounding-box pre-filter, **no native `tgeompoint` storage, no spatial index**. This is the same setup
as `scripts/xengine_compare.py`, here **timed**.

**Why this config exists (keep it separate from the warehouse notebook).**
* **1 vs 2** (this), same open files, MobilityDuck vs MobilityDB -> isolates the **engine**.
* **2 vs 3** (warehouse notebook), same engine, open-no-index vs native-indexed -> isolates the **storage model**.

Queries are copied verbatim from `xengine_compare.py` (answers match Table 7.2). Writes
`results/mobilitydb_lakehouse.csv`.

Requires local PostgreSQL with `postgis + mobilitydb + pg_parquet` reachable via `PGCONN`.

In [1]:
import os, glob, time, statistics
import psycopg2, pandas as pd

PGCONN = "dbname=mdb_lakehouse"    # <-- the MobilityDB database
TZ     = "Europe/Copenhagen"
CLI    = os.getcwd()               # run from benchmark/
L0DIR  = os.path.join(CLI, "data/trips/L0/L0")
GLOB   = f"{L0DIR}/year=2026/month=01/day=*.parquet"   # full month
FILES  = sorted(glob.glob(GLOB)); print(len(FILES), "L0 files")

T0,T1,TMID = "2026-01-15 08:00:00","2026-01-16 08:00:00","2026-01-15 20:00:00"
D0,D1 = "2026-01-15","2026-01-16"
BELT  = "640730.0, 6042487.0, 654100.0, 6058230.0"
N_ITERS, TRIM, WARMUP = 5, 1, 1

31 L0 files


## 1. Load the open Parquet into a heap (bytea traj, **no index**)

In [2]:
pg=psycopg2.connect(PGCONN); pg.autocommit=True; cur=pg.cursor()
cur.execute(f"SET TimeZone='{TZ}';")
for ext in ("postgis","mobilitydb CASCADE","pg_parquet"):
    cur.execute(f"CREATE EXTENSION IF NOT EXISTS {ext};")
cur.execute("DROP TABLE IF EXISTS l0_pq;")
# xmin/xmax collide with Postgres MVCC system columns -> bxmin/bxmax (as in xengine_compare.py)
cur.execute("""CREATE TABLE l0_pq(mmsi bigint, ship_type text, segment_type text,
   traj bytea, tmin timestamptz, tmax timestamptz,
   bxmin float8, bxmax float8, ymin float8, ymax float8, dt date);""")
t=time.time()
for f in FILES: cur.execute(f"COPY l0_pq FROM '{f}' (FORMAT parquet);")
load_s=time.time()-t
cur.execute("SELECT count(*) FROM l0_pq;"); n=cur.fetchone()[0]
cur.execute("SELECT pg_size_pretty(pg_total_relation_size('l0_pq'));"); sz=cur.fetchone()[0]
print(f"loaded {n:,} segments in {load_s:.1f}s   heap size {sz}   (no index)")

loaded 381,713 segments in 164.1s   heap size 4680 MB   (no index)


## 2. The ten queries: decode-on-read, manual bbox filter (verbatim from `xengine_compare.py`)

In [3]:
CLIPPED = f"""
  WITH cand AS (
    SELECT mmsi, ship_type, tgeompointFromEWKB(traj) AS traj FROM l0_pq
    WHERE dt BETWEEN DATE '{D0}' AND DATE '{D1}'
      AND tmax >= TIMESTAMP '{T0}' AND tmin <= TIMESTAMP '{T1}'
      AND bxmax >= 640730.0 AND bxmin <= 654100.0
      AND ymax  >= 6042487.0 AND ymin  <= 6058230.0),
  clipped AS (
    SELECT mmsi, ship_type, g FROM (
      SELECT mmsi, ship_type,
        atStbox(traj, stbox(ST_MakeEnvelope({BELT}), tstzspan('[{T0}, {T1})'))) g
      FROM cand) s WHERE g IS NOT NULL)"""

def prox(margin, agg):
    return f"""{CLIPPED},
  ext AS (SELECT mmsi, g, startTimestamp(g) ts0, endTimestamp(g) ts1,
                 ST_XMin(e) x0, ST_XMax(e) x1, ST_YMin(e) y0, ST_YMax(e) y1
          FROM (SELECT mmsi, g, ST_Envelope(trajectory(g)) e FROM clipped) s WHERE e IS NOT NULL),
  cand2 AS (SELECT a.mmsi m1,b.mmsi m2,a.g t1,b.g t2 FROM ext a JOIN ext b
            ON a.mmsi<b.mmsi AND a.ts0<b.ts1 AND a.ts1>b.ts0
            AND a.x0<=b.x1+{margin} AND b.x0<=a.x1+{margin}
            AND a.y0<=b.y1+{margin} AND b.y0<=a.y1+{margin})
  {agg}"""

QUERIES = {
 "clip_to_region": f"{CLIPPED} SELECT count(DISTINCT mmsi) FROM clipped;",
 "harbour_entry": f"""
   WITH cand AS (SELECT mmsi, atTime(tgeompointFromEWKB(traj), tstzspan('[{T0}, {T1})')) trip FROM l0_pq
     WHERE bxmin<=679171.0 AND bxmax>=666538.0 AND ymin<=6403745.0 AND ymax>=6392057.0
       AND tmin<TIMESTAMP '{T1}' AND tmax>TIMESTAMP '{T0}' AND dt BETWEEN DATE '{D0}' AND DATE '{D1}')
   SELECT count(DISTINCT mmsi) FROM cand
   WHERE trip IS NOT NULL AND eIntersects(trip, ST_MakeEnvelope(666538.0,6392057.0,679171.0,6403745.0));""",
 "both_ports": f"""
   WITH rodby AS (SELECT DISTINCT mmsi FROM l0_pq
       WHERE bxmin<=651422.0 AND bxmax>=651135.0 AND ymin<=6058548.0 AND ymax>=6058230.0
         AND tmin<TIMESTAMP '{T1}' AND tmax>TIMESTAMP '{T0}' AND dt BETWEEN DATE '{D0}' AND DATE '{D1}'
         AND eIntersects(tgeompointFromEWKB(traj), ST_MakeEnvelope(651135.0,6058230.0,651422.0,6058548.0))),
        putt AS (SELECT DISTINCT mmsi FROM l0_pq
       WHERE bxmin<=644896.0 AND bxmax>=644339.0 AND ymin<=6042487.0 AND ymax>=6042108.0
         AND tmin<TIMESTAMP '{T1}' AND tmax>TIMESTAMP '{T0}' AND dt BETWEEN DATE '{D0}' AND DATE '{D1}'
         AND eIntersects(tgeompointFromEWKB(traj), ST_MakeEnvelope(644339.0,6042108.0,644896.0,6042487.0)))
   SELECT count(*) FROM rodby JOIN putt USING(mmsi);""",
 "position_interpolation": f"""{CLIPPED}
   SELECT count(DISTINCT mmsi) FROM (SELECT mmsi, valueAtTimestamp(g, TIMESTAMP '{TMID}') p FROM clipped) s WHERE p IS NOT NULL;""",
 "collision":      prox(300,"SELECT count(*) FROM (SELECT DISTINCT m1,m2 FROM cand2 WHERE nearestApproachDistance(t1,t2)<300) s;"),
 "encounter_zone": prox(500,"SELECT count(*) FROM (SELECT DISTINCT m1,m2 FROM cand2 WHERE nearestApproachDistance(t1,t2)<500) s;"),
 "nearest_approach":prox(2000,"SELECT round(min(nearestApproachDistance(t1,t2))) FROM cand2;"),
 "fleet_summary":  f"{CLIPPED} SELECT round((sum(length(g))/1000.0)::numeric,1) FROM clipped;",
 "bounding_box":   f"""{CLIPPED}
   SELECT round(avg((x1-x0)*(y1-y0)/1e6)::numeric,2) FROM (
     SELECT mmsi, min(x0) x0, max(x1) x1, min(y0) y0, max(y1) y1 FROM (
       SELECT mmsi, ST_XMin(trajectory(g)) x0, ST_XMax(trajectory(g)) x1,
              ST_YMin(trajectory(g)) y0, ST_YMax(trajectory(g)) y1 FROM clipped) s GROUP BY mmsi) t;""",
 "speed_profile":  f"""{CLIPPED}
   SELECT round((percentile_cont(0.5) WITHIN GROUP (ORDER BY vmax)*1.94384)::numeric,1) FROM (
     SELECT mmsi, max(maxValue(speed(g))) vmax FROM clipped GROUP BY mmsi) s;""",
}
ORDER=["clip_to_region","harbour_entry","both_ports","position_interpolation",
       "collision","encounter_zone","nearest_approach","fleet_summary","bounding_box","speed_profile"]
print(len(QUERIES),"queries ready")

10 queries ready


## 3. Time each query (warm, trimmed mean): no index, so expect full-heap scans

In [4]:
def trimmed(xs):
    xs=sorted(xs); xs=xs[TRIM:len(xs)-TRIM] if len(xs)>2*TRIM else xs
    return statistics.mean(xs)
rows=[]
for q in ORDER:
    sql=QUERIES[q]
    for _ in range(WARMUP): cur.execute(sql); cur.fetchall()
    ts=[]; ans=None
    for _ in range(N_ITERS):
        t=time.time(); cur.execute(sql); ans=cur.fetchone()[0]; ts.append(time.time()-t)
    rows.append({"query":q,"answer":ans,"lakehouse_mdb_ms":round(1000*trimmed(ts),1)})
    print(f"{q:24} answer={str(ans):>10}  {1000*trimmed(ts):9.1f} ms")
df=pd.DataFrame(rows); df.to_csv("results/mobilitydb_lakehouse.csv",index=False); df

clip_to_region           answer=       115      112.7 ms


harbour_entry            answer=        99      284.8 ms


both_ports               answer=         4      327.2 ms


position_interpolation   answer=        29      275.1 ms


collision                answer=       163     1621.3 ms


...

nearest_approach         answer=       0.0     1645.2 ms


fleet_summary            answer=    2840.5      242.2 ms


bounding_box             answer=     62.69      791.9 ms


speed_profile            answer=      35.9      315.2 ms


,query,answer,lakehouse_mdb_ms
0,clip_to_region,115,112.7
1,harbour_entry,99,284.8
2,both_ports,4,327.2
3,position_interpolation,29,275.1
4,collision,163,1621.3
5,encounter_zone,194,1604.8
6,nearest_approach,0.0,1645.2
7,fleet_summary,2840.5,242.2
8,bounding_box,62.69,791.9
9,speed_profile,35.9,315.2


**Next:** run `mobilitydb_warehouse.ipynb` (config 3, native + GiST index) on the same data and
window, then compare `results/mobilitydb_lakehouse.csv` (config 2) vs `results/mobilitydb_warehouse.csv`
(config 3) vs your `results/iceberg.csv` (config 1). The three columns give the engine effect and the
storage-model effect separately.